# Laurie's NB

In [2]:
# import crim_intervals and specific modules
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
# import other libraries used in the notebooks
import pandas as pd
from IPython.display import display
from pyvis.network import Network
import altair as alt
import plotly_express as px
import os
import re
import requests
import seaborn as sns

# for use in Jupyter notebook, create a local folder for music files
MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)

else:
    print(MUSDIR, "folder already exists.")

saved_csv folder already exists.
Music_Files folder already exists.


### Get CRIM Metadata from Django

In [7]:
metadata = pd.read_json('https://crimproject.org/data/pieces/')


/var/folders/_s/4t2p1z2x0yxcv068dtqj31tw0000gq/T/ipykernel_91841/3705496990.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  metadata = pd.read_json('https://crimproject.org/data/pieces/')
/var/folders/_s/4t2p1z2x0yxcv068dtqj31tw0000gq/T/ipykernel_91841/3705496990.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  metadata = pd.read_json('https://crimproject.org/data/pieces/')
/var/folders/_s/4t2p1z2x0yxcv068dtqj31tw0000gq/T/ipykernel_91841/3705496990.py:1: FutureWarning: The b

In [8]:
# clean mei link column so we only have one string

metadata['mei_links'] = metadata['mei_links'].str[0]

metadata.head(2)


,url,piece_id,title,full_title,genre,pdf_links,mei_links,composer,date,date_sort,number_of_voices,remarks
0,https://crimproject.org/data/pieces/CRIM_Model...,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,{'url': 'https://crimproject.org/data/genres/m...,[https://crimproject.org/pdf/CRIM_Model_0001.pdf],https://crimproject.org/mei/CRIM_Model_0001.mei,{'url': 'https://crimproject.org/data/people/C...,1538,1538.0,5,
1,https://crimproject.org/data/pieces/CRIM_Model...,CRIM_Model_0002,O gente brunette,O gente brunette,{'url': 'https://crimproject.org/data/genres/c...,[https://crimproject.org/pdf/CRIM_Model_0002.pdf],https://crimproject.org/mei/CRIM_Model_0002.mei,{'url': 'https://crimproject.org/data/people/C...,1548,1548.0,4,


In [12]:
metadata['composerName'] = metadata['composer'].apply(lambda x: x.get('name'))
metadata.head(3)

,url,piece_id,title,full_title,genre,pdf_links,mei_links,composer,date,date_sort,number_of_voices,remarks,composerName
0,https://crimproject.org/data/pieces/CRIM_Model...,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,{'url': 'https://crimproject.org/data/genres/m...,[https://crimproject.org/pdf/CRIM_Model_0001.pdf],https://crimproject.org/mei/CRIM_Model_0001.mei,{'url': 'https://crimproject.org/data/people/C...,1538,1538.0,5,,Johannes Lupi
1,https://crimproject.org/data/pieces/CRIM_Model...,CRIM_Model_0002,O gente brunette,O gente brunette,{'url': 'https://crimproject.org/data/genres/c...,[https://crimproject.org/pdf/CRIM_Model_0002.pdf],https://crimproject.org/mei/CRIM_Model_0002.mei,{'url': 'https://crimproject.org/data/people/C...,1548,1548.0,4,,Thomas Champion
2,https://crimproject.org/data/pieces/CRIM_Model...,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,{'url': 'https://crimproject.org/data/genres/o...,[https://crimproject.org/pdf/CRIM_Model_0003.pdf],https://crimproject.org/mei/CRIM_Model_0003.mei,{'url': 'https://crimproject.org/data/people/C...,11xx,1100.0,1,In solemnitatibus et festis B.M.V.,Anonymous


In [14]:

columns_to_select = ["piece_id", "title", "full_title", "mei_links", 'composerName']
metadata_brief = metadata[columns_to_select]
metadata_brief.head(3)

,piece_id,title,full_title,mei_links,composerName
0,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,https://crimproject.org/mei/CRIM_Model_0001.mei,Johannes Lupi
1,CRIM_Model_0002,O gente brunette,O gente brunette,https://crimproject.org/mei/CRIM_Model_0002.mei,Thomas Champion
2,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,https://crimproject.org/mei/CRIM_Model_0003.mei,Anonymous


In [33]:
metadata_brief.loc[metadata_brief['mei_links'] == 'https://crimproject.org/mei/CRIM_Model_0001.mei'].to_dict()

{'piece_id': {0: 'CRIM_Model_0001'},
 'title': {0: 'Vidi speciosam'},
 'full_title': {0: 'Vidi speciosam'},
 'mei_links': {0: 'https://crimproject.org/mei/CRIM_Model_0001.mei'},
 'composerName': {0: 'Johannes Lupi'}}

In [50]:
corpus_list = ['https://crimproject.org/mei/CRIM_Mass_0014_3.mei', 'https://crimproject.org/mei/CRIM_Model_0009.mei']
list_dfs = []
for url in corpus_list:
    piece = importScore(url)
    mel = piece.melodic().fillna('-')
    df = piece.numberParts(mel)
    temp_dict = metadata_brief.loc[metadata_brief['mei_links'] == url].to_dict()
    for key, value in temp_dict.items():
        df[key] = list(value.values())[0]
    list_dfs.append(df)
output = pd.concat(list_dfs)   

In [51]:
output

,1,2,3,4,piece_id,title,full_title,mei_links,composerName
0.0,Rest,-,Rest,Rest,CRIM_Mass_0014_3,Credo,Missa Mente tota: Credo,https://crimproject.org/mei/CRIM_Mass_0014_3.mei,Antoine de Févin
8.0,Rest,P1,Rest,Rest,CRIM_Mass_0014_3,Credo,Missa Mente tota: Credo,https://crimproject.org/mei/CRIM_Mass_0014_3.mei,Antoine de Févin
12.0,-,-M2,-,-,CRIM_Mass_0014_3,Credo,Missa Mente tota: Credo,https://crimproject.org/mei/CRIM_Mass_0014_3.mei,Antoine de Févin
16.0,-,M2,Rest,Rest,CRIM_Mass_0014_3,Credo,Missa Mente tota: Credo,https://crimproject.org/mei/CRIM_Mass_0014_3.mei,Antoine de Févin
22.0,-,-M2,-,-,CRIM_Mass_0014_3,Credo,Missa Mente tota: Credo,https://crimproject.org/mei/CRIM_Mass_0014_3.mei,Antoine de Févin
...,...,...,...,...,...,...,...,...,...
241.0,-M2,-,-M2,-,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,https://crimproject.org/mei/CRIM_Model_0009.mei,Pierre Cadéac
242.0,P1,-m3,-m2,m3,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,https://crimproject.org/mei/CRIM_Model_0009.mei,Pierre Cadéac
243.0,-,-,-M2,-,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,https://crimproject.org/mei/CRIM_Model_0009.mei,Pierre Cadéac
244.0,-,M2,M2,-m2,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,https://crimproject.org/mei/CRIM_Model_0009.mei,Pierre Cadéac
